In [3]:
import sys
from pathlib import Path

# Add the project root containing 'src' to Python's search path
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / "src").is_dir():
        if str(parent) not in sys.path:
            sys.path.insert(0, str(parent))
        break

import torch
import torch.nn as nn
from torch.optim.lr_scheduler import StepLR
import joblib
import torchaudio
from torchvision.datasets import ImageFolder
from torch.utils.data import random_split
import torchvision.transforms as tt
from collections import Counter
import matplotlib.pyplot as plt
from src.audio.constants import *
from src.audio.logger import logging
from src.audio.exception import CustomException
from src.audio.entity.config_entity import *
from src.audio.entity.artifact_entity import *
from src.audio.entity.custom_model import *
from src.audio.utils import *
from torch.utils.data import DataLoader

In [4]:
class ModelTraining:

    def __init__(self, data_transformation_artifact: DataTransformationArtifacts,
                 model_trainer_config: ModelTrainerConfig):
        self.model_trainer_config = model_trainer_config
        self.data_transformation_artifact = data_transformation_artifact

    def get_data_loader(self, train_data):
        try:
            val_size = int(len(train_data) * 0.2)
            train_size = len(train_data) - val_size

            train_ds, val_ds = random_split(train_data, [train_size, val_size])

            training_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
            valid_dl = DataLoader(val_ds, BATCH_SIZE * 2, num_workers=0, pin_memory=True)

            return training_dl, valid_dl
        except Exception as e:
            raise CustomException(e, sys)

    def get_model(self, train_data):
        try:
            num_classes = len(train_data.classes)
            model = ResNet9(3, num_classes)

            return model
        except Exception as e:
            raise CustomException(e, sys)

    def load_to_gpu(self, training_dl, valid_dl, model):
        try:
            logging.info("loading model to GPU")
            DEVICE = get_default_device()
            model = to_device(model, DEVICE)
            logging.info("model loaded into GPU")
            logging.info("loading data into gpu")

            training_dl = DeviceDataLoader(training_dl, DEVICE)
            valid_dl = DeviceDataLoader(valid_dl, DEVICE)

            return training_dl, valid_dl, model
        except Exception as e:
            raise CustomException(e, sys)

    def train_model(self, model, train_dl, valid_dl):
        try:
            fitted_model, result = my_fit_method(
                epochs=EPOCHS,
                lr=LEARNING_RATE,
                model=model,
                train_data_loader=train_dl,
                val_loader=valid_dl,
                opt_func=OPTIMIZER,
                grad_clip=GRAD_CLIP
            )

            logging.info("Model Training complete")

            return fitted_model, result
        except Exception as e:
            raise CustomException(e, sys)

    def initiate_model_trainer(self):
        try:
            train_transform = tt.Compose([
                tt.Resize(64),
                tt.RandomCrop(64),
                tt.RandomHorizontalFlip(p=0.5),
                tt.ToTensor()
            ])

            os.makedirs(self.model_trainer_config.model_trainer_artifact_dir, exist_ok=True)
            joblib.dump(train_transform, self.model_trainer_config.transformer_object_path)

            train_data = ImageFolder(
                self.data_transformation_artifact.images_folder_path,
                transform=train_transform
            )

            training_dl, valid_dl = self.get_data_loader(train_data)
            model = self.get_model(train_data)

            training_dl, valid_dl, model = self.load_to_gpu(training_dl, valid_dl, model)

            fitted_model, result = self.train_model(
                model=model,
                train_dl=training_dl,
                valid_dl=valid_dl
            )

            logging.info("saving the model")

            torch.save(model.state_dict(), self.model_trainer_config.model_path)

            model_trainer_artifact = ModelTrainerArtifacts(
                model_path=self.model_trainer_config.model_path,
                result=result,
                transformer_object_path=self.model_trainer_config.transformer_object_path
            )

            return model_trainer_artifact

        except Exception as e:
            raise CustomException(e, sys)